# Sparsity Ratio Sweep - Deep Validation

**Why**: Top-k sparsity gives +21pp on sort, +8pp on mazes.

**Goal**: Ratio sweep (0.25/0.5/0.75) x 3 seeds x 2 tasks.

**Hardware**: 1 machine x 8 GPUs (~6h).

In [ ]:
import sys; sys.path.insert(0, '.'); sys.path.insert(0, '..')
from exp_runner import *
import matplotlib.pyplot as plt
%matplotlib inline

## Part A - Prior Results (st08 sparsity0.5)

In [ ]:
df_prior = load_prior()
if df_prior is not None:
    sub = df_prior[(df_prior.stage == 'st08') & (df_prior.sweep == 'sparsity0.5')]
    print(summary_stats(sub))
else:
    print('Prior data not found.')

In [ ]:
if df_prior is not None:
    plot_prior_bar(df_prior, ['sort','mazes'],
                   'st08', 'sparsity0.5', 'Prior: sparsity 0.5 vs baseline',
                   'figures/03_prior_bar.png')

In [ ]:
curves = load_prior_curves()
if curves:
    plot_prior_curves(curves, 'sort',
        [('st00','paper','baseline','#888'),
         ('st08','sparsity0.5','sparsity 0.5','#9467bd')],
        'Sort convergence (prior)', 'figures/03_prior_conv.png')

## Part B - Experiment Design (3 ratios x 3 seeds x 2 tasks = 18 runs)

In [ ]:
exps = make_sparsity(['sort','mazes'], [0,1,2], ratios=[0.25, 0.5, 0.75])
print(f'{len(exps)} experiments')
for e in exps[:6]:
    print(f'  {e.name}')
print('  ...')

In [ ]:
run_all(exps, gpus=8, log_root='logs/deep/03_sparsity', dry_run=True)

## Part C - Run Training

Set `CONFIRM_RUN = True` to launch (~6h).

In [ ]:
CONFIRM_RUN = False
if CONFIRM_RUN:
    done, failed = run_all(exps, gpus=8, log_root='logs/deep/03_sparsity')
else:
    print(f'Set CONFIRM_RUN = True to launch {len(exps)} runs (~6h)')

In [ ]:
status('logs/deep/03_sparsity')

## Part D - Results Analysis

In [ ]:
df = collect('logs/deep/03_sparsity')
if df.empty:
    print('No results yet.')
else:
    print(df[['name','task','best_acc','delta']].to_string(index=False))
    plot_delta_bars(df, 'Sparsity sweep vs baseline', 'figures/03_delta.png')

In [ ]:
if not df.empty:
    import re
    df['ratio'] = df['name'].str.extract(r'sparsity([0-9]+p?[0-9]*)')[0].str.replace('p','.').astype(float)
    plot_sweep_curve(df, 'ratio', title='Sparsity ratio sweep (errorbar = std over seeds)',
                    savepath='figures/03_sweep.png')

In [ ]:
if not df.empty:
    plot_box_seeds(df, 'task', 'best_acc',
                   'Seed variance per task', 'figures/03_box.png')
    print(summary_stats(df, groupby=('task','ratio')))